# Task 10: WGAN-GP (Wasserstein GAN with gradient penalty)

In [1]:
import torch
import torch.nn as nn


In [2]:
class Generator(nn.Module):
    def __init__(self, z_dim=32, out_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, out_dim)
        )
    def forward(self, z):
        return self.net(z)

class Critic(nn.Module):
    def __init__(self, in_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.LeakyReLU(0.2),
            nn.Linear(64, 64), nn.LeakyReLU(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)


In [3]:
def gradient_penalty(critic, real, fake):
    eps = torch.rand(real.size(0), 1)
    interpolated = (eps*real + (1-eps)*fake).requires_grad_(True)
    scores = critic(interpolated)
    grads = torch.autograd.grad(scores, interpolated, grad_outputs=torch.ones_like(scores),
                                 create_graph=True, retain_graph=True)[0]
    return ((grads.norm(2, dim=1) - 1)**2).mean()


In [4]:
G = Generator()
C = Critic()
opt_G = torch.optim.Adam(G.parameters(), lr=1e-4, betas=(0.5, 0.9))
opt_C = torch.optim.Adam(C.parameters(), lr=1e-4, betas=(0.5, 0.9))

real_data = torch.randn(256, 2) * 2 + 3  # toy target distribution
lam = 10


In [5]:
for epoch in range(100):
    for _ in range(5):
        real = real_data[torch.randint(0, 256, (64,))]
        z = torch.randn(64, 32)
        fake = G(z).detach()
        opt_C.zero_grad()
        loss_C = C(fake).mean() - C(real).mean() + lam*gradient_penalty(C, real, fake)
        loss_C.backward()
        opt_C.step()

    z = torch.randn(64, 32)
    fake = G(z)
    opt_G.zero_grad()
    loss_G = -C(fake).mean()
    loss_G.backward()
    opt_G.step()

    if epoch % 20 == 0:
        print(epoch, "loss_C:", loss_C.item(), "loss_G:", loss_G.item())


0 loss_C: 8.336715698242188 loss_G: -0.13914558291435242
20 loss_C: 2.2657320499420166 loss_G: -0.15711264312267303
40 loss_C: -1.7369189262390137 loss_G: -0.26681992411613464
60 loss_C: -3.133040428161621 loss_G: -0.3704266846179962
80 loss_C: -3.7724270820617676 loss_G: -0.40104636549949646
